# Day 94 — Pooling

This notebook covers Spatial Pooling Layers (`nn.MaxPool2d`, `nn.AvgPool2d`, `nn.AdaptiveAvgPool2d`), Zero-Parameter Downsampling, Local Shift Tolerance, Receptive Field Expansion, and Classifier Head Flattening (`.flatten(1)`)

## Validation goals
- Contrast max pooling (`nn.MaxPool2d`), average pooling (`nn.AvgPool2d`), and adaptive global average pooling (`nn.AdaptiveAvgPool2d((1, 1))`) on deterministic and batched feature maps
- Verify zero learnable parameters in pooling layers and trace spatial dimension reductions from `(8, 32, 16, 16)` through `(8, 32, 8, 8)` to `(8, 32, 1, 1)` and flattened `(8, 32)`
- Validate exact numerical pooling outputs on a deterministic $4 \times 4$ tensor grid (`[[7., 8.], [4., 9.]]` for max pooling vs. `[[4., 5.], [1.75, 4.75]]` for average pooling)
- Analyze compute savings ($4\times$ spatial area reduction), receptive field expansion, odd-dimension truncation (`H // 2`), and architectural use cases for max vs. adaptive pooling

### 1. Why Pooling Matters in a CNN

Once a convolutional layer produces feature maps of shape (`N, C, H, W`), keeping full spatial resolution through every layer quickly becomes expensive and overly sensitive to tiny pixel shifts:
- Spatial Downsampling: A standard 2x2 pooling window with stride=2 cuts height and width in half (`H -> H // 2, W -> W // 2`) while preserving the channel count C.
- Local Shift Tolerance: Taking a summary statistic over a small local neighborhood makes representations slightly less sensitive to exact pixel alignment—if a feature shifts by one pixel inside a 2x2 window, the pooled summary often stays similar.
- Zero Learnable Parameters: Unlike `nn.Conv2d` or `nn.Linear`, pooling layers have no weights or biases to train.
- Larger Receptive Field Later On: Shrinking spatial maps early lets deeper convolutional layers cover larger regions of the original image with modest compute.

### 2. Max Pooling vs. Average Pooling vs. Adaptive Pooling

- Max Pooling (`nn.MaxPool2d(kernel_size=2, stride=2)`): Picks the maximum activation inside each window. Strongest when sharp feature presence matters most (such as prominent edge or texture responses after `ReLU`).
- Average Pooling (`nn.AvgPool2d(kernel_size=2, stride=2)`): Computes the mean activation across each window, smoothing values more evenly across the patch.
- Adaptive Average Pooling (`nn.AdaptiveAvgPool2d((1, 1))`): Automatically chooses window/stride sizes to pool each spatial map to a target output size—such as (1, 1) per channel right before a classifier head, regardless of whether the incoming feature map is 4x4 or 8x8.

### 3. Implementing & Inspecting Pooling Layers

Let's write a clean script that applies `MaxPool2d`, `AvgPool2d`, and `AdaptiveAvgPool2d` to a batch of feature maps, verifies shapes and parameter counts, and checks exact values on a small deterministic grid.

In [1]:
import torch
import torch.nn as nn


if __name__ == "__main__":
    torch.manual_seed(42)

    # Exact 1x1x4x4 grid to verify max vs. average pooling by hand
    tiny_grid = torch.tensor(
        [[[[1.0, 3.0, 2.0, 4.0],
           [5.0, 7.0, 6.0, 8.0],
           [2.0, 1.0, 9.0, 3.0],
           [4.0, 0.0, 5.0, 2.0]]]],
        dtype=torch.float32,
    )

    max_pool_2x2 = nn.MaxPool2d(kernel_size=2, stride=2)
    avg_pool_2x2 = nn.AvgPool2d(kernel_size=2, stride=2)
    gap_1x1 = nn.AdaptiveAvgPool2d((1, 1))

    tiny_max = max_pool_2x2(tiny_grid)
    tiny_avg = avg_pool_2x2(tiny_grid)

    # Batch of 8 feature maps: (N=8, C=32, H=16, W=16)
    feats = torch.randn((8, 32, 16, 16), dtype=torch.float32)
    feats_pooled = max_pool_2x2(feats)
    feats_gap = gap_1x1(feats_pooled)

    pool_params = sum(p.numel() for p in max_pool_2x2.parameters())

    print("--- Day 94: Pooling Layers & Spatial Shapes ---")
    print(f"Learnable params in MaxPool2d: {pool_params}\n")

    print("Tiny 4x4 MaxPool2d (2x2, stride=2) -> expected [[7., 8.], [4., 9.]]:")
    print(tiny_max.squeeze(0).squeeze(0))

    print("\nTiny 4x4 AvgPool2d (2x2, stride=2) -> expected [[4., 5.], [1.75, 4.75]]:")
    print(tiny_avg.squeeze(0).squeeze(0))

    print("\nBatch feature map progression:")
    print(f"  Input feats:       {tuple(feats.shape)}")
    print(f"  After MaxPool2d:   {tuple(feats_pooled.shape)}")
    print(f"  After AdaptiveAvg: {tuple(feats_gap.shape)} -> flattened: {tuple(feats_gap.flatten(1).shape)}")

--- Day 94: Pooling Layers & Spatial Shapes ---
Learnable params in MaxPool2d: 0

Tiny 4x4 MaxPool2d (2x2, stride=2) -> expected [[7., 8.], [4., 9.]]:
tensor([[7., 8.],
        [4., 9.]])

Tiny 4x4 AvgPool2d (2x2, stride=2) -> expected [[4., 5.], [1.75, 4.75]]:
tensor([[4.0000, 5.0000],
        [1.7500, 4.7500]])

Batch feature map progression:
  Input feats:       (8, 32, 16, 16)
  After MaxPool2d:   (8, 32, 8, 8)
  After AdaptiveAvg: (8, 32, 1, 1) -> flattened: (8, 32)


### 4. Practical Gotchas When Using Pooling

- Odd Spatial Dimensions: If height or width is odd (for example 7 with kernel_size=2, stride=2), standard pooling drops the last row/column (`7 // 2 = 3`). Watch shapes if you plan to concatenate or add feature maps from different branches later.
- Max Pooling vs. Noisy/Negative Inputs: Because `MaxPool2d` keeps only the largest value per patch, noisy outliers can occasionally dominate a window—which is why `MaxPool2d` is often paired after Conv2d + ReLU or combined with `AdaptiveAvgPool2d` near the head.
- Flattening Before `nn.Linear`: Once spatial dimensions shrink down (for example to `(N, 64, 4, 4) or (N, 64, 1, 1)`), `use x.flatten(start_dim=1)` before passing into a dense classification or regression layer.

### Guided Question:

How many learnable parameters does `nn.MaxPool2d(kernel_size=2, stride=2)` add to a model, and how does it reduce compute in subsequent layers?

### Answer:

`nn.MaxPool2d(kernel_size=2, stride=2)` adds `0` learnable parameters to a model. It performs a fixed deterministic selection independently on every channel.

**How It Reduces Compute in Subsequent Layers**

With kernel_size=2 and stride=2 (and default padding=0), each spatial dimension is halved:$$\text{Out}_{\text{H}} = \left\lfloor \frac{H}{2} \right\rfloor, \quad \text{Out}_{\text{W}} = \left\lfloor \frac{W}{2} \right\rfloor$$

Halving both height and width shrinks the spatial area (H * W) by a factor of 4x per channel:
- Fewer `nn.Conv2d` evaluations: A subsequent convolutional layer evaluates its kernel at one-fourth as many spatial positions per channel, cutting multiply-accumulate operations (`MACs`) and activation memory by roughly 4x (assuming the number of output channels stays constant).
- Smaller flattened `nn.Linear` inputs: If a feature map is flattened before a fully connected layer, a 4x smaller spatial grid cuts the input vector length by 4x, directly reducing weight matrix rows and compute in that linear layer.
- Larger effective receptive field: Downsampling lets subsequent fixed-size kernels cover a wider region of the original input image with fewer layers.

**Trade-Offs to Keep in Mind**

- Information discard: Non-winning activations in each `2 x 2` patch are dropped on the forward pass, and gradients during backpropagation flow only to the winning maximum index in each window.
- Alternative downsampling: Strided convolutions `(stride=2 in nn.Conv2d)` also cut spatial area by 4x while learning how to downsample, at the cost of extra parameters and compute inside that layer.

### Guided Question:

When might `nn.MaxPool2d` be preferable to `nn.AvgPool2d` on feature maps, and when might average/adaptive average pooling be a better fit?

### Answer:

Choosing between `nn.MaxPool2d`, `nn.AvgPool2d`, and `nn.AdaptiveAvgPool2d` depends on whether your task benefits from strongest local feature activation or smooth spatial aggregation and fixed output dimensions.

**When `nn.MaxPool2d` Is Preferable**

`nn.MaxPool2d` passes only the highest activation in each pooling window forward. It is typically preferred when presence and peak response matter more than spatial averaging:
- Sparse or localized features: Edges, corners, keypoints, or small object parts often produce sharp, localized activations inside a larger background. Averaging with surrounding zeros or low background values dilutes that signal; taking the maximum preserves it.
- Early and mid-level feature extraction: Classic `CNNs` often pair `nn.MaxPool2d` with ReLU activations so strong detector responses survive spatial downsampling without being washed out by nearby low-response pixels.
- Minor local shift tolerance: Selecting the strongest response within a small window helps stabilize downstream layers against small pixel shifts while keeping spatial resolution coarse.

**When nn.AvgPool2d Is a Better Fit**

`nn.AvgPool2d` computes the arithmetic mean across each pooling window. It is generally preferred when overall density, smoothness, or full-patch statistics carry the signal:
- Diffuse or distributed features: Textures, lighting gradients, or broad regions where the average intensity across a patch is more informative than a single peak pixel.
- Reducing outlier or noise spikes: Because every element in the window contributes to the mean, a single noisy high activation has less influence than it would under max pooling.
- Smoother gradient flow: During backpropagation, gradients distribute evenly across all cells in the window rather than routing solely through a single maximum index.

**When `nn.AdaptiveAvgPool2d` Is a Better Fit**

`nn.AdaptiveAvgPool2d(output_size)` specifies the desired output spatial size (such as (`1, 1`) for Global Average Pooling) instead of a fixed kernel_size and stride, letting PyTorch compute the pooling window geometry automatically for varying input (`H, W`) sizes:
- Variable input resolutions: When images or feature maps arrive at different heights and widths (or multi-scale inputs) and you need a uniform tensor shape before a classifier or fusion layer.
- Global Average Pooling (`output_size=(1, 1)`): Collapses each channel's entire spatial map into a single summary value, replacing large flattened nn.Linear layers with far fewer parameters and strong spatial regularization.
- Architecture modularity and transferability: Keeps head dimensions consistent across backbones or resolution sweeps without manually recalculating stride and padding formulas at every stage.

### Guided Question:

If a tensor has shape `(16, 64, 28, 28)` and passes through `nn.MaxPool2d(kernel_size=2, stride=2)` followed by `nn.AdaptiveAvgPool2d((1, 1))`, what are the shapes after each operation?

### Answer:

Starting from `(16, 64, 28, 28) (batch size 16, channels 64, height 28, width 28)`:
- After `nn.MaxPool2d(kernel_size=2, stride=2)`: `(16, 64, 14, 14)`
- After `nn.AdaptiveAvgPool2d((1, 1))`: `(16, 64, 1, 1)`

**Step-by-Step Shape Trace**

- After `nn.MaxPool2d(kernel_size=2, stride=2)`:
Batch size (16) and channel count (64) remain unchanged. With kernel_size=2, stride=2, and padding=0, each spatial axis halves from 28 to 14:$$\text{Out} = \left\lfloor \frac{28 + 2(0) - 2}{2} \right\rfloor + 1 = 13 + 1 = 14$$
- After `nn.AdaptiveAvgPool2d((1, 1))`:
Batch size (16) and channel count (64) still stay the same, while spatial dimensions are pooled to the requested (1, 1) output size, producing (16, 64, 1, 1) (often followed by .flatten(1) to get (16, 64) before a linear layer).